In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
"""
Correlación de Spearman entre SUS UX y Seguridad SUS, por grupo (tratamiento),
sobre el subconjunto de respondentes que SÍ contestaron el bloque de
percepción de seguridad (se excluyen quienes no respondieron ninguna de
las 6 preguntas de ese bloque).

Genera:
  - Un PDF combinado con los 4 diagramas de dispersión (uno por grupo)
  - Un PDF individual por grupo (para insertar sueltos en LaTeX)
  - Un Excel con el resumen de rho / p-valor / n por grupo
  - Un Excel con los datos filtrados usados en el análisis

Requisitos:
    pip install pandas openpyxl scipy matplotlib

Uso:
    python correlacion_spearman_filtrado.py
"""

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ------------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------------
ARCHIVO_EXCEL = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"

COL_GRUPO = "Grupo"
COL_SUS_UX = "SUS UX"
COL_SUS_SEG = "seguridad sus"

# Etiquetas en inglés para los gráficos (el artículo está en inglés)
LABEL_SUS_UX = "SUS UX Score"
LABEL_SUS_SEG = "Perceived Security (SUS Score)"

# Preguntas individuales del bloque de percepción de seguridad.
# Un respondente que no contestó NINGUNA de estas se considera "no
# respondió percepción de seguridad" y se excluye del análisis.
PALABRAS_CLAVE_PREGUNTAS_SEGURIDAD = [
    "suficientemente seguro",
    "no lo suficientemente",
    "no es lo suficientemente",
]

CARPETA_SALIDA = "graficos"  # se crea si no existe

import os
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def identificar_columnas_seguridad(df):
    cols = [
        c for c in df.columns
        if any(clave in c for clave in PALABRAS_CLAVE_PREGUNTAS_SEGURIDAD)
    ]
    if not cols:
        raise ValueError("No se encontraron columnas del bloque de percepción de seguridad.")
    return cols


def graficar_grupo(sub, grupo, rho, p_valor, ax=None, standalone_path=None, titulo=None):
    """Dibuja el diagrama de dispersión de un grupo, con rho y p anotados."""
    fig_local = None
    if ax is None:
        fig_local, ax = plt.subplots(figsize=(5.5, 4.5))

    ax.scatter(sub[COL_SUS_UX], sub[COL_SUS_SEG], alpha=0.75, edgecolor="k")
    ax.set_title(titulo if titulo else f"Group {grupo} (n={len(sub)})")
    ax.set_xlabel(LABEL_SUS_UX)
    ax.set_ylabel(LABEL_SUS_SEG)

    texto = f"ρ = {rho:.3f}\np = {p_valor:.3g}"
    ax.text(
        0.05, 0.95, texto,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )

    if fig_local is not None:
        fig_local.tight_layout()
        fig_local.savefig(standalone_path)  # PDF vectorial, listo para LaTeX
        plt.close(fig_local)
        print(f"Gráfico individual guardado en: {standalone_path}")


def main():
    df = repro_data.read_datos_cuanti()

    for col in (COL_GRUPO, COL_SUS_UX, COL_SUS_SEG):
        if col not in df.columns:
            raise ValueError(f"No se encontró la columna '{col}'. Columnas disponibles: {list(df.columns)}")

    cols_seguridad = identificar_columnas_seguridad(df)

    # Excluir a quienes no respondieron ninguna pregunta del bloque de seguridad
    n_total = len(df)
    df_filtrado = df.dropna(how="all", subset=cols_seguridad).copy()
    n_excluidos = n_total - len(df_filtrado)
    print(f"Respondentes totales: {n_total}")
    print(f"Excluidos (sin respuesta en percepción de seguridad): {n_excluidos}")
    print(f"Respondentes analizados: {len(df_filtrado)}\n")

    grupos = sorted(df_filtrado[COL_GRUPO].dropna().unique())

    # --- PDF combinado (2x2) ---
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    axes = axes.flatten()

    resultados = []
    for ax, grupo in zip(axes, grupos):
        sub = df_filtrado[df_filtrado[COL_GRUPO] == grupo].dropna(subset=[COL_SUS_UX, COL_SUS_SEG])
        rho, p_valor = spearmanr(sub[COL_SUS_UX], sub[COL_SUS_SEG])
        resultados.append({"Grupo": grupo, "n": len(sub), "rho_spearman": rho, "p_valor": p_valor})

        graficar_grupo(sub, grupo, rho, p_valor, ax=ax)

        # --- PDF individual por grupo ---
        graficar_grupo(
            sub, grupo, rho, p_valor,
            standalone_path=os.path.join(CARPETA_SALIDA, f"spearman_grupo_{grupo}.pdf"),
        )

    for ax in axes[len(grupos):]:
        ax.set_visible(False)

    fig.suptitle("Spearman Correlation: SUS UX vs. Perceived Security by Group", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    ruta_combinado = os.path.join(CARPETA_SALIDA, "spearman_todos_los_grupos.pdf")
    fig.savefig(ruta_combinado)
    print(f"\nGráfico combinado guardado en: {ruta_combinado}")
    plt.close(fig)

    # Correlación global
    rho_g, p_g = spearmanr(df_filtrado[COL_SUS_UX], df_filtrado[COL_SUS_SEG])
    resultados.append({"Grupo": "Global", "n": len(df_filtrado), "rho_spearman": rho_g, "p_valor": p_g})

    # --- PDF individual de la correlación global (todos los grupos juntos) ---
    graficar_grupo(
        df_filtrado, "Global", rho_g, p_g,
        standalone_path=os.path.join(CARPETA_SALIDA, "spearman_global.pdf"),
        titulo=f"All Groups Combined (n={len(df_filtrado)})",
    )

    resumen = pd.DataFrame(resultados)
    print("\nResumen de correlaciones (subconjunto filtrado):")
    print(resumen.to_string(index=False))

    ruta_resumen = os.path.join(CARPETA_SALIDA, "resumen_correlacion_spearman.xlsx")
    resumen.to_excel(ruta_resumen, index=False)
    print(f"\nResumen exportado a: {ruta_resumen}")

    ruta_datos = os.path.join(CARPETA_SALIDA, "datos_filtrados.xlsx")
    df_filtrado.to_excel(ruta_datos, index=False)
    print(f"Datos filtrados exportados a: {ruta_datos}")


if __name__ == "__main__":
    main()

Respondentes totales: 67
Excluidos (sin respuesta en percepción de seguridad): 7
Respondentes analizados: 60

Gráfico individual guardado en: graficos/spearman_grupo_1.pdf
Gráfico individual guardado en: graficos/spearman_grupo_2.pdf
Gráfico individual guardado en: graficos/spearman_grupo_3.pdf
Gráfico individual guardado en: graficos/spearman_grupo_4.pdf



Gráfico combinado guardado en: graficos/spearman_todos_los_grupos.pdf
Gráfico individual guardado en: graficos/spearman_global.pdf

Resumen de correlaciones (subconjunto filtrado):
 Grupo  n  rho_spearman  p_valor
     1 14      0.606516 0.021472
     2 14      0.849838 0.000119
     3 18      0.534867 0.022191
     4 14     -0.607581 0.021182
Global 60      0.486703 0.000080

Resumen exportado a: graficos/resumen_correlacion_spearman.xlsx
Datos filtrados exportados a: graficos/datos_filtrados.xlsx
